<div style="background: linear-gradient(135deg, #0f172a, #1d4ed8); padding: 24px; border-radius: 16px; color: white; margin-bottom: 20px;">
<h1 style="margin: 0;">Limpeza e preparação — Base 3</h1>
<p style="margin-top: 10px;">Esta etapa prepara a série horária PRSA para as fases de engenharia de atributos e modelagem, sem alterar o significado temporal dos registros.</p>
</div>

**Objetivo:** padronizar timestamps e tipos, validar a grade horária, organizar a estação-alvo e as estações externas e tratar ausências sem vazamento.

**Decisões adotadas:**
- `PM2.5` da estação Aotizhongxin permanece como alvo e **não é imputado**;
- preditores numéricos e categóricos recebem apenas propagação para frente (`forward fill`), usando o último valor observado até a origem;
- não há preenchimento para trás, interpolação bidirecional ou uso de valores futuros;
- valores extremos são preservados e apenas sinalizados;
- a regularização usa a grade horária completa, mas não cria observações sintéticas para o alvo;
- toda medição de preditor será defasada na Fase 3 antes da previsão de `PM2.5(t+1)`.

**Saídas:** `prepared_base3.csv`, `preparation_manifest.json`, `missingness_before_after.csv` e `prepared_schema.csv` em `01_dados/outputs/`.

<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">1. Configuração e contrato temporal</h2>
<p style="margin-bottom:0;">A preparação é determinística. O índice final é uma grade horária, e qualquer preenchimento percorre o tempo apenas no sentido passado → futuro.</p>
</div>

In [9]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_STATE = 42
TARGET = "PM2.5"
FREQUENCY = "h"
HORIZON_HOURS = 1
SOURCE = "https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data"


def find_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "bases" / "grupo3").exists():
            return candidate
    raise FileNotFoundError("Raiz do repositório não encontrada.")


ROOT = find_root(Path.cwd().resolve())
DATA_DIR = ROOT / "bases" / "grupo3"
OUTPUT_DIR = ROOT / "analyses" / "grupo3" / "01_dados" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAIN_PATH = DATA_DIR / "PRSA_Data_Aotizhongxin_20130301-20170228.csv"
EXTERNAL_DIR = DATA_DIR / "externas"

REQUIRED = [
    "No", "year", "month", "day", "hour", "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM", "station",
]
NUMERIC = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
CATEGORICAL = ["wd"]


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">2. Leitura, tipagem e regularização</h2>
<p style="margin-bottom:0;">Cada arquivo é validado contra o schema PRSA. Duplicidades são rejeitadas, não arbitrariamente agregadas. A grade completa é usada para tornar lacunas explícitas.</p>
</div>

In [10]:
def read_raw_station(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    missing_columns = sorted(set(REQUIRED).difference(frame.columns))
    if missing_columns:
        raise ValueError(f"{path.name}: colunas ausentes: {missing_columns}")
    frame = frame[REQUIRED].copy()
    frame["timestamp"] = pd.to_datetime(frame[["year", "month", "day", "hour"]], errors="raise")
    for column in NUMERIC:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame["wd"] = frame["wd"].astype("string")
    frame["station"] = frame["station"].astype("string")
    if frame["timestamp"].duplicated().any():
        raise ValueError(f"{path.name}: timestamps duplicados; nenhuma agregação automática será aplicada.")
    return frame.sort_values("timestamp", kind="stable").set_index("timestamp")


def regularize(frame: pd.DataFrame) -> pd.DataFrame:
    expected = pd.date_range(frame.index.min(), frame.index.max(), freq=FREQUENCY)
    result = frame.drop(columns=["year", "month", "day", "hour", "No"], errors="ignore").reindex(expected)
    result.index.name = "timestamp"
    return result


main_raw = read_raw_station(MAIN_PATH)
external_paths = sorted(EXTERNAL_DIR.glob("PRSA_Data_*.csv"))
external_raw = {
    path.stem.replace("PRSA_Data_", "").rsplit("_20130301-20170228", 1)[0]: read_raw_station(path)
    for path in external_paths
}

main_regular = regularize(main_raw)
external_regular = {name: regularize(frame) for name, frame in external_raw.items()}
expected_index = main_regular.index
if any(not frame.index.equals(expected_index) for frame in external_regular.values()):
    raise ValueError("As estações não compartilham a mesma grade temporal.")

print(f"Grade final: {expected_index.min()} → {expected_index.max()} ({len(expected_index):,} horas)")
print(f"Estações externas: {', '.join(sorted(external_regular))}")


Grade final: 2013-03-01 00:00:00 → 2017-02-28 23:00:00 (35,064 horas)
Estações externas: Changping, Guanyuan, Wanliu


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">3. Imputação causal e preservação do alvo</h2>
<p style="margin-bottom:0;">O <code>forward fill</code> replica somente a última medição conhecida. O alvo local e os indicadores de ausência não são preenchidos. Valores extremos não são removidos nem winsorizados.</p>
</div>

In [11]:
def causal_fill(frame: pd.DataFrame, station_name: str, target_station: bool = False) -> tuple[pd.DataFrame, dict[str, int]]:
    result = frame.copy()
    before = result.isna().sum()
    fillable = [column for column in NUMERIC + CATEGORICAL if column in result.columns]
    if target_station:
        fillable.remove(TARGET)
    for column in fillable:
        result[column] = result[column].ffill()
    after = result.isna().sum()
    changes = {}
    for column in result.columns:
        filled = int(before.get(column, 0) - after.get(column, 0))
        if filled:
            changes[f"{station_name}.{column}"] = filled
    return result, changes


main_prepared, main_fills = causal_fill(main_regular, "Aotizhongxin", target_station=True)
external_prepared = {}
fill_counts = dict(main_fills)
for station_name, frame in external_regular.items():
    external_prepared[station_name], station_fills = causal_fill(frame, station_name)
    fill_counts.update(station_fills)

# A ausência inicial não recebe valor artificial e será tratada como indisponibilidade.
assert main_prepared[TARGET].isna().sum() == main_regular[TARGET].isna().sum()
assert all(value >= 0 for value in fill_counts.values())
print("Preenchimentos causais realizados:")
display(pd.Series(fill_counts, name="quantidade_preenchida").to_frame())


Preenchimentos causais realizados:


,quantidade_preenchida
Aotizhongxin.PM10,718
Aotizhongxin.SO2,935
Aotizhongxin.NO2,1023
Aotizhongxin.CO,1776
Aotizhongxin.O3,1719
Aotizhongxin.TEMP,20
Aotizhongxin.PRES,20
Aotizhongxin.DEWP,20
Aotizhongxin.RAIN,20
Aotizhongxin.wd,81


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">4. Consolidação e auditoria das transformações</h2>
<p style="margin-bottom:0;">A tabela final mantém o alvo local, os preditores locais e as medições das estações externas com prefixos explícitos. Os atributos temporais são conhecidos e serão usados posteriormente conforme o horizonte aprovado.</p>
</div>

In [12]:
prepared = pd.DataFrame(index=expected_index)
prepared["target_pm25"] = main_prepared[TARGET]
for column in NUMERIC + CATEGORICAL:
    if column != TARGET:
        prepared[f"local_{column}"] = main_prepared[column]
for station_name, frame in external_prepared.items():
    for column in NUMERIC + CATEGORICAL:
        prepared[f"{station_name}_{column}"] = frame[column]

prepared = prepared.reset_index()
prepared["year"] = prepared["timestamp"].dt.year.astype("int16")
prepared["month"] = prepared["timestamp"].dt.month.astype("int8")
prepared["day"] = prepared["timestamp"].dt.day.astype("int8")
prepared["hour"] = prepared["timestamp"].dt.hour.astype("int8")
prepared["dayofweek"] = prepared["timestamp"].dt.dayofweek.astype("int8")
prepared["is_weekend"] = prepared["dayofweek"].isin([5, 6]).astype("int8")

# Flags são informativas: indicam ausência no dado bruto antes do ffill.
prepared["target_pm25_missing_raw"] = main_regular[TARGET].isna().to_numpy()
for column in NUMERIC + CATEGORICAL:
    if column != TARGET:
        prepared[f"local_{column}_missing_raw"] = main_regular[column].isna().to_numpy()
for station_name, frame in external_regular.items():
    for column in NUMERIC + CATEGORICAL:
        prepared[f"{station_name}_{column}_missing_raw"] = frame[column].isna().to_numpy()

assert prepared["timestamp"].is_monotonic_increasing
assert prepared["timestamp"].duplicated().sum() == 0
assert prepared["target_pm25_missing_raw"].sum() == 925
assert prepared["target_pm25"].isna().sum() == 925

prepared_path = OUTPUT_DIR / "prepared_base3.csv"
prepared.to_csv(prepared_path, index=False)

before_counts = pd.Series({
    "target_pm25": int(main_regular[TARGET].isna().sum()),
    **{f"local_{column}": int(main_regular[column].isna().sum()) for column in NUMERIC + CATEGORICAL if column != TARGET},
    **{f"{station}_{column}": int(frame[column].isna().sum()) for station, frame in external_regular.items() for column in NUMERIC + CATEGORICAL},
}, name="ausencias_antes")
after_counts = pd.Series({
    "target_pm25": int(main_prepared[TARGET].isna().sum()),
    **{f"local_{column}": int(main_prepared[column].isna().sum()) for column in NUMERIC + CATEGORICAL if column != TARGET},
    **{f"{station}_{column}": int(frame[column].isna().sum()) for station, frame in external_prepared.items() for column in NUMERIC + CATEGORICAL},
}, name="ausencias_depois")
missingness = pd.concat([before_counts, after_counts], axis=1).fillna(0).astype(int)
missingness["preenchidas_causalmente"] = missingness["ausencias_antes"] - missingness["ausencias_depois"]
missingness.to_csv(OUTPUT_DIR / "missingness_before_after.csv", index_label="variavel")

def dtype_role(column: str) -> str:
    if column == "timestamp":
        return "índice temporal"
    if column == "target_pm25":
        return "alvo não imputado"
    if column.endswith("_missing_raw"):
        return "indicador de ausência bruta"
    if column in {"year", "month", "day", "hour", "dayofweek", "is_weekend"}:
        return "calendário conhecido"
    return "preditor imputado causalmente" if column not in {"target_pm25"} else "alvo"

schema = pd.DataFrame({
    "coluna": prepared.columns,
    "dtype": [str(prepared[column].dtype) for column in prepared.columns],
    "papel": [dtype_role(column) for column in prepared.columns],
})
schema.to_csv(OUTPUT_DIR / "prepared_schema.csv", index=False)

manifest = {
    "source": SOURCE,
    "input_files": [str(MAIN_PATH.relative_to(ROOT))] + [str(path.relative_to(ROOT)) for path in external_paths],
    "output_file": str(prepared_path.relative_to(ROOT)),
    "target": "target_pm25",
    "target_source": "Aotizhongxin.PM2.5",
    "rows": int(len(prepared)),
    "columns": int(prepared.shape[1]),
    "time_start": str(prepared["timestamp"].min()),
    "time_end": str(prepared["timestamp"].max()),
    "frequency": "1 hour",
    "duplicates_removed": 0,
    "missing_slots_added": 0,
    "missing_target_preserved": int(prepared["target_pm25"].isna().sum()),
    "imputation": {
        "method": "forward fill only",
        "direction": "past to future",
        "target_imputed": False,
        "predictors_imputed": True,
        "initial_missing_values_preserved": True,
        "backfill": False,
        "interpolation": False,
    },
    "outliers": {
        "action": "preserved and flagged in Phase 1; no removal or winsorization",
        "iqr_used_for_removal": False,
    },
    "feature_engineering_deferred": True,
    "random_state": RANDOM_STATE,
    "forecast_horizon_hours": HORIZON_HOURS,
    "transformations": [
        "timestamp constructed from year/month/day/hour",
        "numeric and categorical types standardized",
        "records sorted chronologically",
        "all stations reindexed to the complete hourly grid",
        "raw missingness flags retained",
        "predictor missingness filled causally with ffill",
        "target PM2.5 left unimputed",
        "station columns prefixed to avoid ambiguity",
        "calendar columns added",
    ],
}
with (OUTPUT_DIR / "preparation_manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

display(missingness.sort_values("ausencias_antes", ascending=False).head(20))
print(f"Base preparada: {prepared_path}")
print(f"Shape final: {prepared.shape}")


,ausencias_antes,ausencias_depois,preenchidas_causalmente
Wanliu_O3,2107,0,2107
Wanliu_CO,1812,0,1812
local_CO,1776,0,1776
Guanyuan_CO,1753,0,1753
local_O3,1719,0,1719
Changping_CO,1521,0,1521
Guanyuan_O3,1173,0,1173
Wanliu_NO2,1070,0,1070
local_NO2,1023,0,1023
local_SO2,935,0,935


Base preparada: /Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/analyses/grupo3/01_dados/outputs/prepared_base3.csv
Shape final: (35064, 103)


<div style="background:#dbeafe; border-radius: 12px; padding: 16px; color:#1e3a8a;">
<h2 style="margin-top:0;">Validação da Fase 2</h2>
<ul style="margin-bottom:0;">
<li>A grade final deve ser estritamente horária, ordenada e sem duplicidades.</li>
<li>O número de ausências do alvo deve permanecer inalterado.</li>
<li>Preditores só podem ser preenchidos por observações anteriores ou no mesmo instante.</li>
<li>Valores extremos não são excluídos nesta fase.</li>
<li>A seleção de lags, janelas e demais features fica para <code>03_feature_engineering.ipynb</code>.</li>
</ul>
</div>